# Stage 5 — Evaluate + export ONNX (self-contained)
Rebuilds model + test loader from the cached crops, evaluates on the held-out
test split, exports versioned ONNX, writes model card.

In [1]:
import sys; sys.path.insert(0,'/workspace/lib')
import os, json, time
import numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES=True
from sklearn.metrics import classification_report, f1_score
import dataset_lib as D
from dataset_lib import CLASSES

DATASET_ROOT='/dataset'
MODEL_VERSION=D.latest_manifest_version(DATASET_ROOT)
MODEL_DIR=f'/dataset/models/{MODEL_VERSION}'
CROPS_ROOT=f'/dataset/crops/{MODEL_VERSION}'
IMG_SIZE=518; BATCH=128; MEAN=(0.485,0.456,0.406); STD=(0.229,0.224,0.225)
device='cuda' if torch.cuda.is_available() else 'cpu'
print('Evaluating',MODEL_VERSION,'on',device)

Evaluating v1.1 on cuda


In [2]:
# rebuild test loader from cached crops
class ResizePadSquare:
    def __init__(s,sz): s.sz=sz
    def __call__(s,im):
        w,h=im.size; sc=s.sz/max(w,h); nw,nh=round(w*sc),round(h*sc)
        im=im.resize((nw,nh),Image.BILINEAR); c=Image.new('RGB',(s.sz,s.sz),(0,0,0))
        c.paste(im,((s.sz-nw)//2,(s.sz-nh)//2)); return c
eval_tf=transforms.Compose([ResizePadSquare(IMG_SIZE),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
name_to_canon={n:i for i,n in enumerate(CLASSES)}
test_ds=datasets.ImageFolder(f'{CROPS_ROOT}/test',transform=eval_tf)
idx_map={i:name_to_canon[n] for i,n in enumerate(test_ds.classes)}
test_ds.samples=[(p,idx_map[t]) for (p,t) in test_ds.samples]; test_ds.targets=[idx_map[t] for t in test_ds.targets]
test_dl=DataLoader(test_ds,batch_size=BATCH,num_workers=8,pin_memory=True)
print('test crops:',len(test_ds))

test crops: 5766


In [3]:
# rebuild model + load trained weights
model=timm.create_model('vit_large_patch14_dinov2.lvd142m',pretrained=False,num_classes=len(CLASSES),img_size=IMG_SIZE)
model.load_state_dict(torch.load(f'{MODEL_DIR}/best.pt',map_location=device))
model.to(device).eval()
print('loaded',f'{MODEL_DIR}/best.pt')

/tmp/ipykernel_1501/3477389817.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'{MODEL_DIR}/best.pt',map_location=device))


loaded /dataset/models/v1.1/best.pt


In [4]:
# evaluate on held-out test split
ys,ps=[],[]
with torch.no_grad():
    for x,y in test_dl:
        x=x.to(device)
        with torch.autocast('cuda',dtype=torch.bfloat16,enabled=(device=='cuda')): o=model(x)
        ps+=o.argmax(1).cpu().tolist(); ys+=y.tolist()
rep=classification_report(ys,ps,target_names=CLASSES,digits=3,zero_division=0)
macro_f1=f1_score(ys,ps,average='macro',zero_division=0)
print(rep)
print('macro-F1:',round(macro_f1,4))

                      precision    recall  f1-score   support

  ErinaceusEuropaeus      1.000     1.000     1.000       220
 SciurusCarolinensis      1.000     0.992     0.996       250
     SciurusVulgaris      0.990     1.000     0.995       206
  CapreolusCapreolus      0.996     0.978     0.987       232
       CervusElaphus      0.973     0.982     0.978       222
        VulpesVulpes      0.992     0.996     0.994       263
          MelesMeles      1.000     0.996     0.998       223
              Person      0.991     1.000     0.996       229
         CapraHircus      1.000     0.972     0.986       142
OryctolagusCuniculus      1.000     1.000     1.000       169
     ColumbaPalumbus      1.000     0.994     0.997       172
  PhasianusColchicus      0.994     0.989     0.992       181
           OvisAries      0.985     1.000     0.992       198
    PasserDomesticus      1.000     0.994     0.997       178
            DamaDama      0.967     0.981     0.974       210
       

In [5]:
# export versioned ONNX with metadata
import onnx
dummy=torch.randn(1,3,IMG_SIZE,IMG_SIZE,device=device)
onnx_path=f'{MODEL_DIR}/deepfaune_uk_{MODEL_VERSION}.onnx'
torch.onnx.export(model,dummy,onnx_path,input_names=['input'],output_names=['logits'],
    dynamic_axes={'input':{0:'batch'},'logits':{0:'batch'}},opset_version=17,do_constant_folding=True)
m=onnx.load(onnx_path)
for k,v in [('classes',json.dumps(CLASSES)),('input_size',str(IMG_SIZE)),
            ('mean',json.dumps(list(MEAN))),('std',json.dumps(list(STD))),
            ('version',MODEL_VERSION),('macro_f1',str(round(macro_f1,4)))]:
    p=m.metadata_props.add(); p.key=k; p.value=v
onnx.save(m,onnx_path); print('exported',onnx_path)

/opt/conda/lib/python3.11/site-packages/torch/__init__.py:2040: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


exported /dataset/models/v1.1/deepfaune_uk_v1.1.onnx


In [6]:
# model card + report
man=D.load_manifest(DATASET_ROOT,MODEL_VERSION)
card={'version':MODEL_VERSION,'macro_f1_test':round(macro_f1,4),
      'n_train_images':man['split_counts'].get('train'),'classes':CLASSES,
      'class_counts':man['class_counts']}
json.dump(card,open(f'{MODEL_DIR}/model_card.json','w'),indent=2)
open(f'{MODEL_DIR}/test_report.txt','w').write(rep+f'\nmacro-F1: {macro_f1:.4f}\n')
print('wrote model_card.json + test_report.txt to',MODEL_DIR)

wrote model_card.json + test_report.txt to /dataset/models/v1.1
